In [1]:
import Pkg
Pkg.activate("..")
include("../src/NS_Numerics.jl")
using .NS_Numerics

  Activating 

Using dtype = IntervalArithmetic

project at `~/Changhe/julia code/residual_verification`


.Interval{Float64}
Number of threads = 56


# Eigenpair Error Analysis

This notebook rigorously evaluates the accuracy of the computed eigenpair $(\lambda, v)$ of the linearized operator.

The main objectives are to obtain rigorous upper bounds for:

- The residual:
  
  $$
  L v - \lambda v
  $$

- The adjoint operator applied to the eigenfunction:

  $$
  L^T v
  $$

- The eigenfunction itself and its derivatives.

We compute:

- Rigorous $L^\infty$ and Sobolev $W^{k,\infty}$ norms using interval arithmetic.
- Rigorous $L^2$ norms and normalized error magnitudes.

These quantities are used to verify the accuracy of the eigenpair and to quantify the residual error relative to the eigenfunction size.

All bounds in this notebook are computed rigorously using interval arithmetic and validated numerical operators.

In [2]:
# Load the self-similar profile U from data file.
# This profile defines the background state of the linearized operator.
UP = from_mat_file("../data/UP.mat", 2; domain="frequency")
M0, M1 = size(UP["u3"].freq_func)

# Load the eigenfunction v and its corresponding eigenvalue λ.
# This eigenpair was previously computed numerically and is now verified rigorously.
up, lbd = from_mat_file("../data/up_eig.mat", 1; domain="frequency")
println("eigenvalue: ", lbd)

eigenvalue: [-0.113142, -0.113142]_com


In [3]:
# We construct grids covering the full spatial domain in order to obtain
# rigorous upper bounds for Sobolev W^{k,∞} norms.

# Domain size in β and θ directions
L0, L1 = Pi * dt(2), Pi / dt(2)
# Number of grid points in each direction
N0, N1 = 16000, 2000
# Sobolev regularity order
k = 10

# Construct uniform grids in β and θ
# These are used to evaluate functions and operators rigorously.
β_pt = dt.(0:N0) .*(L0/dt(N0))
θ_pt = dt.(0:N1) .*(L1/dt(N1))

# Construct operators
# These represent:
#   Linear operator:       L
#   Identity operator:     I
#   Adjoint operator:      Lᵀ
lin_mat = get_operator_matrix(up, β_pt, θ_pt, "linear")
I_mat = get_operator_matrix(up, β_pt, θ_pt, "identity")
lin_T_mat = get_operator_matrix(up, β_pt, θ_pt, "linear"; transpose=true)
grad_mat = get_operator_matrix(up, β_pt, θ_pt, "gradient")
I_phy_mat = get_operator_matrix(up, β_pt, θ_pt, "identity"; weight_flag=false)

# Compute the transport terms
U_grad_u = mul_A_gradB(UP, up, β_pt, θ_pt)
u_grad_U, u_grad_T_U = mul_A_gradB(up, UP, β_pt, θ_pt; transpose=true)

# Compute the residuals and gradients
# Need the L^{2} norms of these terms: L^T v, L v - λ v and v
I_up = apply(I_mat, up) 
err = U_grad_u + u_grad_U - apply(lin_mat, up) - I_up * lbd
eig_T_up = u_grad_T_U - U_grad_u - apply(lin_T_mat, up)

# Need the L^{∞} norms of these terms: v and \nabla v
grad_up = apply(grad_mat, up; new_keys=["A11", "A12", "A13", "A21", "A22", "A23", "A31", "A32", "A33"])
I_phy_up = apply(I_phy_mat, up; new_keys=["u1", "u2", "u3"]);

In [4]:
freq_lst = dt.([2 * M0 + 10, 2 * M1 + 10])

# Store all relevant vectors whose norms must be verified
vec_dict = Dict("err" => err, "I_up" => I_up, "eig_T_up" => eig_T_up, "grad_up" => grad_up, "I_phy_up" => I_phy_up)
norm_dict = Dict()

# Compute rigorous L^∞ and W^{k,∞} norms using interval arithmetic.
# Higher Sobolev norms are required for rigorous derivative bounds.
for key in keys(vec_dict)
    mode = key in ["eig_T_up", "I_up", "err"] ? "W$(k+2)Inf" : "LInf"
    factor = dt(key in ["eig_T_up", "err"] ? 2 : 1)
    norm_dict[key] = norm_lst(vec_dict[key], mode; paras=freq_lst .* factor, domain=[L0,L1])
end
# We cannot print the results yet since v is not L^2 normalized yet.

In [5]:
# Refine Sobolev bounds using extended grids.
# This improves rigor and ensures accurate finite-difference derivative estimates.
L0, L1 = Pi / dt(2), Pi / dt(2)
N0, N1 = 12000, 6000

# Extend grids to include ghost points needed for finite-difference stencils
β_pt = dt.(-k:N0+k) .*(L0/dt(N0))
θ_pt = dt.(-k:N1+k) .*(L1/dt(N1))

lin_mat = get_operator_matrix(up, β_pt, θ_pt, "linear")
I_mat = get_operator_matrix(up, β_pt, θ_pt, "identity")

U_grad_u = mul_A_gradB(UP, up, β_pt, θ_pt)
u_grad_U = mul_A_gradB(up, UP, β_pt, θ_pt)

# Only recompute the residual term
I_up = apply(I_mat, up) 
err = U_grad_u + u_grad_U - apply(lin_mat, up) - I_up * lbd;

In [6]:
# Refine the Sobolev bound for the residual term using optimized parameters.
# This step improves tightness of the rigorous upper bound.
paras_lst = Linf_to_paras(norm_dict["err"], k, freq_lst .* dt(2))
err_norms = norm_lst(err, "W$(k)Inf"; paras=paras_lst, domain=[L0,L1])
norm_dict["err"] = err_norms;

In [7]:
# Compute L² norms of all relevant quantities.
# These norms are used to quantify residual error relative to eigenfunction size.
L0, L1 = Pi / dt(2), Pi / dt(2)
N0, N1 = 36000, 18000

β_pt = dt.(0:N0) .*(L0/dt(N0))
θ_pt = dt.(0:N1) .*(L1/dt(N1));

# Compute nonlinear interaction terms
u_grad_U, u_grad_T_U = mul_A_gradB(up, UP, β_pt, θ_pt; transpose=true)
U_grad_u = mul_A_gradB(UP, up, β_pt, θ_pt)

lin_mat = get_operator_matrix(up, β_pt, θ_pt, "linear")
lin_T_mat = get_operator_matrix(up, β_pt, θ_pt, "linear"; transpose=true)
I_mat = get_operator_matrix(up, β_pt, θ_pt, "identity")

# Need the L^{2} norms of these terms: L^T v, L v - λ v and v
I_up = apply(I_mat, up) 
err = U_grad_u + u_grad_U - apply(lin_mat, up) - I_up * lbd
eig_T_up = u_grad_T_U - U_grad_u - apply(lin_T_mat, up);

In [8]:
# Compute normalized L² norms.
# We normalize all quantities by ||v||₂ so that residual size can be interpreted
# relative to eigenfunction magnitude.

I_fac = 0
vec_dict = Dict("err" => err, "I_up" => I_up, "eig_T_up" => eig_T_up)
for key in ["I_up", "err", "eig_T_up"]
    term = vec_dict[key]
    L2_norms = norm_lst(term, "L2"; paras=norm_dict[key])

    if key == "I_up"
        I_fac = L2_norms
        # Compute the L2 norm of v and use it as a normalizing factor
    else
        println("L^2 norm of ", key, ": ", L2_norms / I_fac)
    end
end

L^2 norm of err: [1.65374e-5, 1.86929e-5]_trv
L^2 norm of eig_T_up: [6.38757, 6.38757]_trv


In [ ]:
# Print L^∞ norm of v and ∇ v after normalization.
println("L^{∞} norm of I_phy_up: ", norm(norm_dict["I_phy_up"] / I_fac))
println("L^{∞} norm of grad_up: ", norm(norm_dict["grad_up"] / I_fac))

L^{∞} norm of I_phy_up: [2.51136, 2.51136]_trv
L^{∞} norm of grad_up: [3.39973, 3.39973]_trv
